# NB0 — Tính toán Normalization Stats Toàn cục (2020 - 2024)

**Mục đích:** # Tính toán giá trị Trung bình (Mean) và Độ lệch chuẩn (Std) của toàn bộ dữ liệu thuộc tập Huấn luyện (Train: 2020-2024) cho cả ERA5 và NASA.
Những giá trị này sẽ được dùng làm hệ quy chiếu chung để chuẩn hóa cho tất cả các năm (bao gồm cả Val 2025 và Test 2026), giúp mô hình không bị sai lệch (data leakage).

**Lưu ý:**
Do lượng dữ liệu 5 năm rất lớn, script này được thiết kế để tính toán từng biến (variable-by-variable) nhằm tối ưu RAM trên Kaggle.

In [1]:
!pip install zarr xarray dask netCDF4 h5netcdf -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.6/319.6 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 60.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 56.3 MB/s eta 0:00:00


In [2]:
import xarray as xr
import numpy as np
import os, gc, json, glob

# ── ĐƯỜNG DẪN INPUT ──────────────────────────────────────────────────
BASE_ERA5 = "/kaggle/input/notebooks/b22dccn320trnminhhiu/fork-of-nb1-era5-2020-2026"
BASE_NASA = "/kaggle/input/notebooks/b22dccn320trnminhhiu/fork-of-nb2-nasa"
OUT = "/kaggle/working"

# Chỉ định rõ các năm thuộc tập Train
TRAIN_YEARS = [2020, 2021, 2022, 2023, 2024]

# Chunk tối ưu để load 5 năm không bị đầy RAM
ERA5_LOAD_CHUNKS = {'time': 200}
NASA_LOAD_CHUNKS = {'time': 200}

print(f"✓ Sẵn sàng tính Stats cho tập Train: {TRAIN_YEARS}")

✓ Sẵn sàng tính Stats cho tập Train: [2020, 2021, 2022, 2023, 2024]


## 1. Load và Gộp Dữ liệu ERA5 (2020 - 2024)

In [3]:
print("--- Quét và Load dữ liệu ERA5 (Lazy) ---")

pressure_paths = []
accum_paths = []
instant_paths = []

for year in TRAIN_YEARS:
    pressure_paths.extend(glob.glob(f"{BASE_ERA5}/era5_data/year_{year}/era5_pressure_*.nc"))
    accum_paths.extend(glob.glob(f"{BASE_ERA5}/era5_data/year_{year}/*accum.nc"))
    instant_paths.extend(glob.glob(f"{BASE_ERA5}/era5_data/year_{year}/*instant.nc"))

# Mở tập dữ liệu đa năm
ds_pressure = xr.open_mfdataset(sorted(pressure_paths), combine='by_coords', chunks=ERA5_LOAD_CHUNKS, engine='h5netcdf')
ds_accum    = xr.open_mfdataset(sorted(accum_paths), combine='by_coords', chunks=ERA5_LOAD_CHUNKS, engine='h5netcdf')
ds_instant  = xr.open_mfdataset(sorted(instant_paths), combine='by_coords', chunks=ERA5_LOAD_CHUNKS, engine='h5netcdf')

# Đổi tên biến xung đột
if 'z' in ds_instant.data_vars: ds_instant = ds_instant.rename({'z': 'geopotential_surf'})
if 'z' in ds_accum.data_vars:   ds_accum = ds_accum.rename({'z': 'geopotential_surf_acc'})

# Bỏ tọa độ dư thừa
JUNK_COORDS = ['expver', 'step', 'number']
def drop_junk(ds):
    drop = [v for v in JUNK_COORDS if v in ds.coords or v in ds.data_vars]
    return ds.drop_vars(drop, errors='ignore')

ds_pressure = drop_junk(ds_pressure)
ds_accum = drop_junk(ds_accum)
ds_instant = drop_junk(ds_instant)

# Gộp dữ liệu
ds_brain_train = xr.merge([ds_pressure, ds_accum, ds_instant], join='inner')

rename_map = {}
if 'valid_time' in ds_brain_train.coords: rename_map['valid_time'] = 'time'
if 'latitude'   in ds_brain_train.coords: rename_map['latitude']   = 'lat'
if 'longitude'  in ds_brain_train.coords: rename_map['longitude']  = 'lon'
if rename_map:
    ds_brain_train = ds_brain_train.rename(rename_map)

print(f"✓ Đã gộp thành công ds_brain_train.")
print(f"  Tổng số Timesteps: {len(ds_brain_train.time)}")
print(f"  Thời gian: {str(ds_brain_train.time.values[0])[:10]} -> {str(ds_brain_train.time.values[-1])[:10]}")

--- Quét và Load dữ liệu ERA5 (Lazy) ---
✓ Đã gộp thành công ds_brain_train.
  Tổng số Timesteps: 7308
  Thời gian: 2020-01-01 -> 2024-12-31


## 2. Tính toán Stats cho ERA5 (Cuốn chiếu từng biến để chống tràn RAM)

In [4]:
print("--- Tính toán Mean và Std cho ERA5 (Sẽ mất thời gian) ---")

stats_dict = {}
total_vars = len(ds_brain_train.data_vars)

for i, var in enumerate(ds_brain_train.data_vars, 1):
    print(f"[{i}/{total_vars}] Đang tính toán cho biến: {var}...")
    
    # Rút trích đúng biến đang tính toán
    da = ds_brain_train[var]
    
    # Tính mean và std trên trục không gian và thời gian
    mean_val = da.mean(dim=['time', 'lat', 'lon']).compute()
    std_val  = da.std(dim=['time', 'lat', 'lon']).compute()
    
    # Chống chia cho 0 nếu std quá nhỏ
    std_val = std_val.where(std_val > 1e-8, other=1e-6)
    
    # Phân loại biến 2D (Bề mặt) và 3D (Phân tầng áp suất)
    if 'pressure_level' not in da.dims:
        m = float(mean_val.values)
        s = float(std_val.values)
        stats_dict[var] = {'mean': m, 'std': s}
        print(f"      -> Bề mặt: Mean={m:.4f}, Std={s:.4f}")
    else:
        # Lặp qua từng mức áp suất
        for lev_idx, lev in enumerate(mean_val.pressure_level.values):
            m = float(mean_val.values[lev_idx])
            s = float(std_val.values[lev_idx])
            key = f"{var}_{int(lev)}hpa"
            stats_dict[key] = {'mean': m, 'std': s}
        print(f"      -> 3D Level: Đã lưu {len(mean_val.pressure_level)} tầng áp suất.")
    
    # Dọn RAM ngay lập tức
    del da, mean_val, std_val
    gc.collect()

# Lưu kết quả ra file JSON
with open(f'{OUT}/era5_normalization_stats.json', 'w') as f:
    json.dump(stats_dict, f, indent=2)

print(f"\n✓ HOÀN TẤT ERA5! Đã lưu: {OUT}/era5_normalization_stats.json")

--- Tính toán Mean và Std cho ERA5 (Sẽ mất thời gian) ---
[1/12] Đang tính toán cho biến: z...
      -> 3D Level: Đã lưu 2 tầng áp suất.
[2/12] Đang tính toán cho biến: q...
      -> 3D Level: Đã lưu 2 tầng áp suất.
[3/12] Đang tính toán cho biến: u...
      -> 3D Level: Đã lưu 2 tầng áp suất.
[4/12] Đang tính toán cho biến: v...
      -> 3D Level: Đã lưu 2 tầng áp suất.
[5/12] Đang tính toán cho biến: tp...
      -> Bề mặt: Mean=0.0003, Std=0.0007
[6/12] Đang tính toán cho biến: u10...
      -> Bề mặt: Mean=-1.2343, Std=3.8931
[7/12] Đang tính toán cho biến: v10...
      -> Bề mặt: Mean=-0.0259, Std=3.3926
[8/12] Đang tính toán cho biến: d2m...
      -> Bề mặt: Mean=295.5115, Std=4.3853
[9/12] Đang tính toán cho biến: t2m...
      -> Bề mặt: Mean=299.2100, Std=4.1821
[10/12] Đang tính toán cho biến: msl...
      -> Bề mặt: Mean=101083.8047, Std=393.2068
[11/12] Đang tính toán cho biến: geopotential_surf...
      -> Bề mặt: Mean=1448.6569, Std=4086.6250
[12/12] Đang tính toán cho biến:

## 3. Load Dữ liệu và Tính Stats cho NASA (2020 - 2024)

In [5]:
print("--- Quét và Load dữ liệu NASA (Lazy) ---")

nasa_paths = []
for year in TRAIN_YEARS:
    nasa_paths.extend(glob.glob(f"{BASE_NASA}/nasa_ir_data/year_{year}/*.nc4"))

ds_nasa_train = xr.open_mfdataset(sorted(nasa_paths), chunks=NASA_LOAD_CHUNKS, combine='by_coords')

if 'Tb' not in ds_nasa_train.data_vars and 'IR' in ds_nasa_train.data_vars:
     ds_nasa_train = ds_nasa_train.rename({'IR': 'Tb'})

print(f"✓ Đã nạp thành công ds_nasa_train.")
print(f"  Tổng số Timesteps: {len(ds_nasa_train.time)}")

--- Quét và Load dữ liệu NASA (Lazy) ---
✓ Đã nạp thành công ds_nasa_train.
  Tổng số Timesteps: 14616


In [6]:
print("--- Tính toán Mean và Std cho NASA ---")

# Tính mean và std toàn cục cho biến Tb
nasa_mean = float(ds_nasa_train['Tb'].mean().compute())
nasa_std  = float(ds_nasa_train['Tb'].std().compute())

if nasa_std < 1e-6:
    print(f"⚠️ Cảnh báo: NASA std quá nhỏ ({nasa_std})")
else:
    print(f"  Tb Train Mean: {nasa_mean:.2f} K")
    print(f"  Tb Train Std : {nasa_std:.2f} K")

# Lưu kết quả vào NetCDF file
stats_dataset = xr.Dataset({
    'mean': xr.DataArray(nasa_mean), 
    'std': xr.DataArray(nasa_std)
})
stats_dataset.to_netcdf(f'{OUT}/nasa_normalization_stats.nc')

print(f"✓ HOÀN TẤT NASA! Đã lưu: {OUT}/nasa_normalization_stats.nc")

--- Tính toán Mean và Std cho NASA ---
  Tb Train Mean: 275.57 K
  Tb Train Std : 23.72 K
✓ HOÀN TẤT NASA! Đã lưu: /kaggle/working/nasa_normalization_stats.nc


## 4. Tổng kết

In [7]:
print("=" * 50)
print("DANH SÁCH FILE STATS ĐÃ TẠO")
print("=" * 50)

for file_name in ['era5_normalization_stats.json', 'nasa_normalization_stats.nc']:
    file_path = f"{OUT}/{file_name}"
    if os.path.exists(file_path):
        size_kb = os.path.getsize(file_path) / 1024
        print(f"✓ {file_name:<35s} | {size_kb:.2f} KB")
    else:
        print(f"❌ THIẾU FILE: {file_name}")

print("\n=> HƯỚNG DẪN TIẾP THEO:")
print("1. Tải 2 file trên về máy cá nhân (hoặc sử dụng tính năng 'Save Version' của Kaggle).")
print("2. Upload 2 file này lên Kaggle thành 1 Dataset mới (ví dụ: 'climate-norm-stats').")
print("3. Add Dataset đó vào Notebook NB3 (Template theo năm) và đổi đường dẫn STATS_DIR.")

DANH SÁCH FILE STATS ĐÃ TẠO
✓ era5_normalization_stats.json       | 1.26 KB
✓ nasa_normalization_stats.nc         | 2.84 KB

=> HƯỚNG DẪN TIẾP THEO:
1. Tải 2 file trên về máy cá nhân (hoặc sử dụng tính năng 'Save Version' của Kaggle).
2. Upload 2 file này lên Kaggle thành 1 Dataset mới (ví dụ: 'climate-norm-stats').
3. Add Dataset đó vào Notebook NB3 (Template theo năm) và đổi đường dẫn STATS_DIR.
